### Post analysis script ratio extractions
This notebook is the final notebook, and should be followed by the `Automated_analysis.ipynb` notebook. 
This script will sort the files in cell wise manner, for each cell, the total count of localizations are found for each channel and used to extract the occupation ratio of multimer complexes. Run each cell individually. The bottom two cells are only relevant if you want to look at dual colour analysis. 

User guide:
1. Select the folder containing the 'multimer_summary' files. 
    - The script is currently look for the `multimer_summary` mark to identify the files needed for analysis.
    - If output_path is `None`, the script will create a new folder in the input folder called Cell_ratios
2. Select the parameters for the analysis:
    - `max_multimer` : The maximum size of the multimer complex to be included in the analysis.
    - `remove_above_max` : If True, the dataset will be pruned to only include the multimer complexes with a size equal or less to the `max_multimer` parameter. If False, the dataset will sum all the multimer complexes above the `max_multimer` parameter and add to the multimer_size of `max_multimer`.
    - `percentage_decimals` : The number of decimals to be used in the percentage output, if None, the percentage will not be rounded 
3. Run the script.

In [62]:
from automation_utils2 import * 
files_path = '/Users/Katie/Library/CloudStorage/OneDrive-TheFrancisCrickInstitute/EGFR Alix collab project/2026-06-18_palm_repeat4/DNM2 no egf/Cells/2026-06-18_DNM2_NOEGF'

output_path = None # If none will be saved in the same folder as the input files under {files_path}/Cell_ratio/*


max_multimer = 50
remove_above_max = True
percentage_decimals = None



grp_files = group_cell_area_files_by_cell(os.path.join(files_path, "Multimers" ), path_ending='multimer_summary.csv')
pivot_table_group = group_cell_area_files_by_cell(os.path.join(files_path, "Pivot_table"), path_ending = 'pivot_table.csv')

print(f'Found files for {len(grp_files)} cells')

Found files for 5 cells


In [63]:
if output_path is None:
    output_path = os.path.join(files_path, 'Cell_ratios')

os.makedirs(output_path, exist_ok=True)

collective_list = []
pivot_list = []

for cell, files in grp_files.items():
    df_list = collect_multimer_files(files, max_multimer, remove_above_max)
    df = pd.concat(df_list, axis=0)
    df_transposed = analyise_complete_cell_df(df, output_path, cell, percentage_decimals)
    collective_list.append(df_transposed)

for cell, files in pivot_table_group.items():
    df = cellect_pivot_files(files)
    percentage = find_percentage_from_pivot(df)
    pivot_ratio = create_ratio_from_channel_0(percentage)

    pivot_list.append(pivot_ratio)
    pivot_ratio.to_csv(os.path.join(output_path, f'Pivot_table_cell_{cell}.csv'))

if len(pivot_list) > 0:
    key_ordering = list(pivot_table_group.keys())
    filtered_pivot = [pivot.query('multimer_complex <= @max_multimer & multimer_complex > 2') for pivot in pivot_list]
    collective_df = pd.concat(filtered_pivot, axis=0, keys=key_ordering)
    grp_pivot = collective_df.groupby('multimer_complex')
    final_pivot = grp_pivot.apply(lambda x: get_receptor_ratio_for_cells(x, cell))
    final_pivot.to_csv(os.path.join(output_path, 'Pivot_table_all_cells.csv'))

In [ ]:
import os
import pandas as pd
from collections import defaultdict

# ============================================================
# PATHS
# ============================================================
MULTIMER_FOLDER = os.path.join(files_path, "Multimers")
PIVOT_FOLDER   = os.path.join(files_path, "Pivot_table")
OUTPUT_FOLDER  = os.path.join(files_path, "Cell_ratio", "heteromer_summary")
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ============================================================
#  SETTINGS
# ============================================================
def to_percent(x):
    if percentage_decimals is None:
        return x * 100
    else:
        return round(x * 100, percentage_decimals)

# ============================================================
#  LOADERS
# ============================================================
def load_multimer_summary_df(path):
    return pd.read_csv(path)

def extract_heteromer_composition(path):
    df = pd.read_csv(path, index_col=0)
    df.index   = df.index.astype(int)
    df.columns = df.columns.astype(int)
    het = []
    for ch1 in df.index:
        for ch0 in df.columns:
            count = df.loc[ch1, ch0]
            if ch0>0 and ch1>0 and count>0:
                het.append({
                    "ch0": ch0,
                    "ch1": ch1,
                    "count": int(count),
                    "total": ch0 + ch1
                })
    return pd.DataFrame(het) if het else pd.DataFrame(columns=["ch0","ch1","count","total"])

# ============================================================
# GROUP FILES BY CELL
# ============================================================
cells = defaultdict(lambda: {"multimer": [], "pivot": []})

for cid, flist in grp_files.items():
    for f in flist:
        if f.endswith("multimer_summary.csv"):
            cells[cid]["multimer"].append(f)

for cid, flist in pivot_table_group.items():
    for f in flist:
        if f.endswith("pivot_table.csv"):
            cells[cid]["pivot"].append(f)

all_het_rows = []

# ============================================================
# MAIN LOOP
# ============================================================
receptor_rows = []
channel_summary_rows = []

cell_total_map = {}
total_ch0_map = {}
total_ch1_map = {}

for cell, fdict in cells.items():

    # ---------- 1. HOMOMERS ----------
    mult = pd.concat([load_multimer_summary_df(f) for f in fdict["multimer"]])
    mult = mult.groupby("multimer_size").sum().reset_index()

    if remove_above_max:
        mult = mult[mult["multimer_size"] <= max_multimer]

    mult["ch0_hom"] = mult["channel_0"] * mult["multimer_size"]
    mult["ch1_hom"] = mult["channel_1"] * mult["multimer_size"]

    # ---------- 2. HETEROMERS ----------
    het_df = pd.concat([extract_heteromer_composition(f) for f in fdict["pivot"]],
                       ignore_index=True)

    if remove_above_max:
        het_df = het_df[het_df["total"] <= max_multimer]

    if len(het_df) > 0:
        het_df["ch0_r"] = het_df["ch0"] * het_df["count"]
        het_df["ch1_r"] = het_df["ch1"] * het_df["count"]
        het_df["heteromer_receptors"] = het_df["ch0_r"] + het_df["ch1_r"]

        het_df2 = het_df.copy()
        het_df2["cell"] = cell
        all_het_rows.append(het_df2)
    else:
        het_df = pd.DataFrame(columns=["ch0","ch1","count","total","ch0_r","ch1_r","heteromer_receptors"])

    # ---------- 3. CELL TOTALS ----------
    ch0_hom_total = mult["ch0_hom"].sum()
    ch1_hom_total = mult["ch1_hom"].sum()
    ch0_het_total = het_df["ch0_r"].sum()
    ch1_het_total = het_df["ch1_r"].sum()

    total_receptors = ch0_hom_total + ch1_hom_total + ch0_het_total + ch1_het_total

    cell_total_map[cell] = total_receptors
    total_ch0_map[cell] = ch0_hom_total + ch0_het_total
    total_ch1_map[cell] = ch1_hom_total + ch1_het_total

    # ---------- 4. PER-CHANNEL SUMMARY ----------
    channel_summary_rows.append({
        "cell": cell,
        "channel_0_total": total_ch0_map[cell],
        "channel_1_total": total_ch1_map[cell],
        "channel_0_homomer_percent":    to_percent(ch0_hom_total / total_ch0_map[cell]),
        "channel_0_heteromer_percent":  to_percent(ch0_het_total / total_ch0_map[cell]),
        "channel_1_homomer_percent":    to_percent(ch1_hom_total / total_ch1_map[cell]),
        "channel_1_heteromer_percent":  to_percent(ch1_het_total / total_ch1_map[cell]),
    })

    # ---------- 5. PER-SIZE TABLE ----------
    # homomers
    for _, row in mult.iterrows():
        receptor_rows.append({
            "cell": cell,
            "multimer_size": int(row["multimer_size"]),
            "ch0_homomer_receptors": row["ch0_hom"],
            "ch1_homomer_receptors": row["ch1_hom"],
            "ch0_heteromer_receptors": 0,
            "ch1_heteromer_receptors": 0
        })

    # heteromers
    for _, row in het_df.iterrows():
        receptor_rows.append({
            "cell": cell,
            "multimer_size": int(row["total"]),
            "ch0_homomer_receptors": 0,
            "ch1_homomer_receptors": 0,
            "ch0_heteromer_receptors": row["ch0_r"],
            "ch1_heteromer_receptors": row["ch1_r"]
        })

# ============================================================
# 6. AGGREGATE & COMPUTE ALL % COLUMNS
# ============================================================
receptor_df = pd.DataFrame(receptor_rows)
agg = receptor_df.groupby(["cell","multimer_size"]).sum()

agg["total_receptors"] = agg.index.get_level_values("cell").map(cell_total_map)
agg["ch0_total"]       = agg.index.get_level_values("cell").map(total_ch0_map)
agg["ch1_total"]       = agg.index.get_level_values("cell").map(total_ch1_map)

# ---------- CORRECT per-size homomer %
agg["ch0_homomer_percent_within_ch0_multimersize"] = (
    agg["ch0_homomer_receptors"] / agg["ch0_total"]
).apply(to_percent)

agg["ch1_homomer_percent_within_ch1_multimersize"] = (
    agg["ch1_homomer_receptors"] / agg["ch1_total"]
).apply(to_percent)

# ---------- CORRECT per-size heteromer %
agg["ch0_heteromer_percent_within_ch0_multimersize"] = (
    agg["ch0_heteromer_receptors"] / agg["ch0_total"]
).apply(to_percent)

agg["ch1_heteromer_percent_within_ch1_multimersize"] = (
    agg["ch1_heteromer_receptors"] / agg["ch1_total"]
).apply(to_percent)

# ---------- TOTAL homomer/heteromer %
agg["ch0_homomer_percent_within_ch0"] = (
    agg.groupby("cell")["ch0_homomer_receptors"].transform("sum") / agg["ch0_total"]
).apply(to_percent)

agg["ch1_homomer_percent_within_ch1"] = (
    agg.groupby("cell")["ch1_homomer_receptors"].transform("sum") / agg["ch1_total"]
).apply(to_percent)

agg["ch0_heteromer_percent_within_ch0"] = (
    agg.groupby("cell")["ch0_heteromer_receptors"].transform("sum") / agg["ch0_total"]
).apply(to_percent)

agg["ch1_heteromer_percent_within_ch1"] = (
    agg.groupby("cell")["ch1_heteromer_receptors"].transform("sum") / agg["ch1_total"]
).apply(to_percent)

# ============================================================
# 7. SAVE OUTPUTS
# ============================================================
agg.reset_index().to_csv(
    os.path.join(OUTPUT_FOLDER,"receptor_level_multimer_distribution.csv"),
    index=False
)

pd.DataFrame(channel_summary_rows).to_csv(
    os.path.join(OUTPUT_FOLDER,"per_channel_proportions.csv"),
    index=False
)

# Save heteromer stoichiometry for Script 2
if len(all_het_rows) > 0:
    het_all = pd.concat(all_het_rows)
else:
    het_all = pd.DataFrame(columns=["cell","ch0","ch1","count","total","ch0_r","ch1_r","heteromer_receptors"])

het_all.to_csv(os.path.join(OUTPUT_FOLDER,"heteromer_stoichiometry_raw.csv"), index=False)

print("✅ Script v3 complete.")
print("✅ All homomer + heteromer % columns restored correctly.")
print("✅ heteromer_stoichiometry_raw.csv saved.")


KeyError: 'channel_0'

In [ ]:
import pandas as pd
import os
from math import gcd

HET_FOLDER = os.path.join(files_path, "Cell_ratio", "heteromer_summary")
INPUT = os.path.join(HET_FOLDER, "heteromer_stoichiometry_raw.csv")
OUTPUT = os.path.join(HET_FOLDER, "heteromer_type_summary.csv")

# Load
het = pd.read_csv(INPUT)

if het.empty or het["count"].sum() == 0:
    print("⚠️ No heteromers detected — saving empty summary.")
    pd.DataFrame(columns=["cell","type","cluster_count","total_clusters","percent_of_heteromers"])\
        .to_csv(OUTPUT, index=False)
else:
    het = het[het["count"] > 0]

    # reduce ratios
    def reduce_ratio(a,b):
        a=int(a); b=int(b)
        if a==0: return f"0:{b}"
        if b==0: return f"{a}:0"
        g=gcd(a,b)
        return f"{a//g}:{b//g}"

    het["type"] = het.apply(lambda r: reduce_ratio(r["ch0"], r["ch1"]), axis=1)

    summary = (
        het.groupby(["cell","type"])["count"]
           .sum()
           .rename("cluster_count")
           .reset_index()
    )

    totals = summary.groupby("cell")["cluster_count"].sum().rename("total_clusters")
    summary = summary.merge(totals, on="cell")

    summary["percent_of_heteromers"] = (
        100 * summary["cluster_count"] / summary["total_clusters"]
    )

    summary.to_csv(OUTPUT, index=False)
    print("✅ heteromer_type_summary.csv saved")